In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used

# Energy emission peaks in MeV
Cs137_MeV = 0.662
Na22_MeV = [0.511, 1.275, 1.786]
Co60_MeV = [1.173, 1.332, 2,505]
Am241_MeV = 0.0595
# Background emission
K40_MeV = 1.460
Tl208_MEV = 2.614

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kViolet, ROOT.kYellow, ROOT.kSpring, ROOT.kCyan,]

In [2]:
files = sorted((project_root/'data/260506').glob('SUBT_*'))
acqs = [SiphraAcquisition(file) for file in files]
#print(acqs)

bg_na_acq = acqs[0]
#led_na_acqs = [acqs[3], acqs[4], acqs[5], acqs[1], acqs[6], acqs[7], acqs[2]] #6 and 7 double peaks
led_na_acqs = [acqs[3], acqs[4], acqs[5], acqs[1], acqs[2]]
print(led_na_acqs)

bg_nosrc_acq = acqs[15]
led_nosrc_acqs = [acqs[13], acqs[12], acqs[10], acqs[11], acqs[9], acqs[8], acqs[14]]
print(led_nosrc_acqs)

[SIPHRAAcquisition(File: 'SUBT_04_LEDTesting_200Hz_77ns'), SIPHRAAcquisition(File: 'SUBT_05_LEDTesting_200Hz_78ns'), SIPHRAAcquisition(File: 'SUBT_06_LEDTesting_200Hz_79ns'), SIPHRAAcquisition(File: 'SUBT_02_LEDTesting_200Hz_80ns'), SIPHRAAcquisition(File: 'SUBT_03_LEDTesting_200Hz_83ns')]
[SIPHRAAcquisition(File: 'SUBT_14_LEDTesting_200Hz_78ns'), SIPHRAAcquisition(File: 'SUBT_13_LEDTesting_200Hz_79ns'), SIPHRAAcquisition(File: 'SUBT_11_LEDTesting_200Hz_80ns'), SIPHRAAcquisition(File: 'SUBT_12_LEDTesting_200Hz_80,5ns'), SIPHRAAcquisition(File: 'SUBT_10_LEDTesting_200Hz_81ns'), SIPHRAAcquisition(File: 'SUBT_09_LEDTesting_200Hz_81,5ns'), SIPHRAAcquisition(File: 'SUBT_15_LEDTesting_200Hz_82ns')]


In [3]:
# histograms
hists = {}
sources = []

bg = bg_na_acq
filepath = bg.filepath.stem
src_bg = (MetadataLoader.load(bg.metadataFile)).source
print(src_bg)
sources.append(src_bg)
hist_bg = ROOT.TH1F(f"{src_bg} Background", "", BITS12, 0 , BITS12)
hist_bg.Fill(bg['s']/len(bg.active_chs))
hist_bg.GetXaxis().SetTitle("ADC channel number")
hist_bg.GetYaxis().SetTitle(r"Normalized counts")
hists[f"{src_bg}_BG"] = hist_bg

for sgnl in (led_na_acqs):
    filepath = sgnl.filepath.stem
    src = (MetadataLoader.load(sgnl.metadataFile)).source
    #src = src.split(" ", 1)[1]
    print(src)
    sources.append(src)
    hist_sgnl = ROOT.TH1F(f"{src} Signal", "", BITS12, 0, BITS12)
    hist_sgnl.Fill(sgnl['s']/len(sgnl.active_chs))
    hist_sgnl.Scale(bg.exposure/sgnl.exposure)
    hist_sgnl.GetXaxis().SetTitle("ADC channel number")
    hist_sgnl.GetYaxis().SetTitle(r"Normalized counts")
    hists[src] = hist_sgnl

    del hist_sgnl
del hist_bg

Na-22
Na-22 LED200Hz77ns
Na-22 LED200Hz78ns
Na-22 LED200Hz79ns
Na-22 LED200Hz80ns
Na-22 LED200Hz83ns


In [4]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 1600, 1200)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lg = ROOT.TLegend(0.48, 0.61, 0.75, 0.83)


for idx, (src) in enumerate(sources): 
    if idx == 0:
        sgnl = hists[src_bg+'_BG']
        lg.AddEntry(sgnl, "Background (Na-22)")
        sgnl.SetLineColor(colors[0])
        sgnl.Draw("hist")

    else:
        sgnl = hists[src]
        lg.AddEntry(sgnl, "Na-22 + " + src)
        sgnl.SetLineColor(colors[idx])
        sgnl.SetTitle(src)
        sgnl.Draw("sames hist")
    ROOT.gPad.Update()
    cv.SetLogy()

lg.Draw()
cv.Draw()

In [7]:
from analysis import *

#Calibration based on the 3 peaks of the Na-22 spectrum


hist = hists[src_bg+'_BG']
energy_ranges = [(60, 140), (200, 260)]
energies = Na22_MeV[:2]

c_fit = calibration_fit(hist, energy_ranges, energies)
print(c_fit)

[0.005814974542718861, -0.08427400626613428]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      964.052
NDf                       =           77
Edm                       =  6.46974e-08
NCalls                    =           73
Constant                  =      4005.73   +/-   12.1232     
Mean                      =      102.369   +/-   0.0548556   
Sigma                     =      20.9615   +/-   0.0576411    	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =       61.851
NDf                       =           57
Edm                       =  1.38378e-06
NCalls                    =           96
Constant                  =        871.6   +/-   5.89847     
Mean                      =      233.754   +/-   0.275453    
Sigma                     =      29.4463   +/-   0.484563     	 (limited)
****************************************
Minimizer is Linear / Migrad
Chi2               

In [8]:
#Calibrating LED histograms based on Na22 calibration fit
hists_cal = []

hist_cal_na = calibrated_histogram(c_fit, bg_na_acq, BITS12)
hist_cal_na.Scale(1/bg_na_acq.exposure)
hists_cal.append(hist_cal_na)

for acq in led_na_acqs:
    hist_cal = calibrated_histogram(c_fit, acq, BITS12)
    hist_cal.Scale(1/acq.exposure)
    hists_cal.append(hist_cal)
    del hist_cal 


# Plot the LED histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1000, 800)
lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)

print(sources)
for idx, (src, hist_cal) in enumerate(zip(sources, hists_cal)):
    #print(hist_cal)
    hist_cal.SetLineColor(colors[idx])
    hist_cal.GetXaxis().SetTitle("Energy [MeV]")
    hist_cal.GetYaxis().SetTitle(r"Count rate [Hz]")
    lg1.AddEntry(hist_cal, src, "l")
    if idx == 0:
        hist_cal.Draw("hist")
    else:
        hist_cal.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()
    

#Calculate energy resolution
peaks = [(2.9, 3.6), (4.6, 5.6), (7.3, 8.4), (16.2, 17.4), (18.7, 19.6) ]
peak_energies = []
resolutions = []
    
for i in range(1, len(hists_cal)):
    hist = hists_cal[i]
    if i == 0:
        res = energy_resolution(hist, [(0.13, 0.6), (1.1, 1.5), (1.5 , 2.1)],  Na22_MeV)
    else:
        cal_fit=ROOT.TF1("cal_fit_" + str(i), "gaus", peaks[i-1][0], peaks[i-1][1])
        hist.Fit(cal_fit, "R+S")
        peak_energy = cal_fit.GetParameter(1)
        print(peak_energy)
        res = energy_resolution(hist, [peaks[i-1]], [peak_energy])
    resolutions.append(res)
print(resolutions)

['Na-22', 'Na-22 LED200Hz77ns', 'Na-22 LED200Hz78ns', 'Na-22 LED200Hz79ns', 'Na-22 LED200Hz80ns', 'Na-22 LED200Hz83ns']
3.3056240077108656
5.1067178120799435
7.846072398696955
16.879005240458568
19.188214866014622
[[0.12205104564233057], [0.0825929821157529], [0.06305159865614277], [0.030167527812108262], [0.021422714616928525]]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      204.272
NDf                       =          135
Edm                       =   1.3618e-07
NCalls                    =           70
Constant                  =      2.13012   +/-   0.0114275   
Mean                      =      3.30562   +/-   0.000830202 
Sigma                     =     0.171318   +/-   0.000728968  	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      204.272
NDf                       =          135
Edm                       =   1.3618e-07
NCalls                    =           70
C

Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
